<a href="https://colab.research.google.com/github/omarelmalak/csc490-group2/blob/omar-optimize-logo-detector/Logo_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8 Logo Detection End-to-End Pipeline

### Step 1: Environment Setup and Library Imports

In [1]:
# Install Ultralytics library for YOLOv8 training
!pip install -q ultralytics

import os
import shutil
import glob
import yaml
import random
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
from ultralytics import YOLO

print("Environment ready and libraries imported!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment ready and libraries imported!


### Step 2: Mount Google Drive
Run this cell to link your Google Drive so the script can access your zipped dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3: Extract Logo Dataset
Extracts your zipped dataset directly from Google Drive into the local session storage.

In [63]:
# Unzip the logo dataset from Google Drive into /content/logos
!unzip -q "/content/drive/MyDrive/logodet.zip" -d "/content/logos"

# Inspect the extracted directory structure to verify it matches what we need
print("Extracted contents of /content/logos:")
for root, dirs, files in os.walk("/content/logos"):
    level = root.replace("/content/logos", '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:3]:  # Print up to 3 files as preview
        print(f"{subindent}{f}")
    if len(files) > 3:
        print(f"{subindent}... and {len(files) - 3} more files")

Streaming output truncated to the last 5000 lines.
                ... and 93 more files
            Freia/
                58.xml
                77.xml
                10.jpg
                ... and 137 more files
            laura secord chocolates/
                43.xml
                58.xml
                77.xml
                ... and 123 more files
            shengfeng/
                38.xml
                38.jpg
                60.xml
                ... and 13 more files
            coffeeheaven/
                10.jpg
                69.jpg
                60.xml
                ... and 35 more files
            Cows/
                59.jpg
                62.xml
                13.xml
                ... and 7 more files
            chenkeming-1/
                33.jpg
                38.xml
                30.jpg
                ... and 29 more files
            maarud/
                43.xml
                58.xml
                10.jpg
                ... and 107 mo

### Step 4: Convert Pascal XML Annotations to YOLO TXT Format
Parses the XML annotations in the "Other" folder and converts them to standard normalized YOLO `.txt` coordinate files next to each image.

In [82]:
# --- CONFIGURATION ---
SOURCE_XML_DIR = "/content/logos/LogoDet-3K/Others"

# 1. First, discover all folders (classes) in Others to build a robust, deterministic class list
# This ensures the IDs never drift between XML parsing and YAML generation.
raw_classes = sorted([d for d in os.listdir(SOURCE_XML_DIR) if os.path.isdir(os.path.join(SOURCE_XML_DIR, d))])

class_to_id = {name: i for i, name in enumerate(raw_classes)}
print(f"Deterministic mapping created for {len(raw_classes)} directories.")

def convert_coords(size, box):
    dw = 1.0 / size[0] if size[0] > 0 else 1.0
    dh = 1.0 / size[1] if size[1] > 0 else 1.0
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

# Convert each XML to TXT inside its respective directory
xml_files = glob.glob(os.path.join(SOURCE_XML_DIR, "**/*.xml"), recursive=True)
converted_count = 0
for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        if w == 0 or h == 0:
            continue

        txt_filename = os.path.splitext(os.path.basename(xml_file))[0] + ".txt"
        txt_output_path = os.path.join(os.path.dirname(xml_file), txt_filename)

        with open(txt_output_path, 'w') as out_file:
            for obj in root.iter('object'):
                cls_name = obj.find('name').text
                # Use the immediate parent folder name as the fallback ground truth class
                parent_folder = os.path.basename(os.path.dirname(xml_file))

                # Resolve class ID using the directory name to ensure alignment
                cls_id = class_to_id.get(parent_folder, class_to_id.get(cls_name, 0))

                xmlbox = obj.find('bndbox')
                b = (
                    float(xmlbox.find('xmin').text),
                    float(xmlbox.find('xmax').text),
                    float(xmlbox.find('ymin').text),
                    float(xmlbox.find('ymax').text)
                )
                bb = convert_coords((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{coord:.6f}' for coord in bb])}\n")
        converted_count += 1
    except Exception as e:
        pass

print(f"Successfully converted {converted_count} XML files to aligned YOLO TXT format!")

Deterministic mapping created for 371 directories.
Successfully converted 15513 XML files to aligned YOLO TXT format!


### Step 5: Define Paths & Split Dataset (80% Train, 10% Val, 10% Test)
This section processes only the "Other" folder, matches the images and converted `.txt` label files, and distributes them proportionally.

In [83]:
import os
import glob
import random
import shutil
from collections import defaultdict

# Configuration
SOURCE_IMAGES_DIR = "/content/logos/LogoDet-3K/Others"
DATASET_ROOT = "/content/logo_yolo_dataset"

# 1. Clean and recreate directory structures completely
if os.path.exists(DATASET_ROOT):
    shutil.rmtree(DATASET_ROOT)

for split_type in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DATASET_ROOT, 'images', split_type), exist_ok=True)
    os.makedirs(os.path.join(DATASET_ROOT, 'labels', split_type), exist_ok=True)

# 2. Gather image files recursively
all_images = []
valid_exts = ('.png', '.jpg', '.jpeg')
for root, dirs, files in os.walk(SOURCE_IMAGES_DIR):
    for f in files:
        if f.lower().endswith(valid_exts):
            all_images.append(os.path.join(root, f))

# 3. Match image-label pairs grouped by brand folder
class_to_pairs = defaultdict(list)
for img_path in all_images:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    img_dir = os.path.dirname(img_path)
    label_path = os.path.join(img_dir, f"{base_name}.txt")
    class_name = os.path.basename(img_dir)
    if os.path.exists(label_path):
        class_to_pairs[class_name].append((img_path, label_path))

# 4. Perform robust split and copy with class-prefixed unique filenames
segmented_count = 0
total_copied = 0

for class_name, pairs in class_to_pairs.items():
    random.seed(42)
    random.shuffle(pairs)

    n_samples = len(pairs)
    if n_samples == 0:
        continue

    val_idx = int(n_samples * 0.8)
    test_idx = int(n_samples * 0.9)

    train_pairs = pairs[:val_idx]
    val_pairs = pairs[val_idx:test_idx]
    test_pairs = pairs[test_idx:]

    if len(val_pairs) == 0 and n_samples > 1:
        val_pairs = [pairs[-1]]
        train_pairs = pairs[:-1]

    def copy_files_unique(pairs_list, split_name):
        global total_copied
        for img, lbl in pairs_list:
            img_ext = os.path.splitext(img)[1]
            base = os.path.splitext(os.path.basename(img))[0]

            # We prefix the filename with the class name to make it totally unique
            unique_name = f"{class_name}_{base}"

            dest_img = os.path.join(DATASET_ROOT, 'images', split_name, f"{unique_name}{img_ext}")
            dest_lbl = os.path.join(DATASET_ROOT, 'labels', split_name, f"{unique_name}.txt")

            shutil.copy2(img, dest_img)
            shutil.copy2(lbl, dest_lbl)
            total_copied += 1

    copy_files_unique(train_pairs, 'train')
    copy_files_unique(val_pairs, 'val')
    copy_files_unique(test_pairs, 'test')

    segmented_count += 1

print(f"=== Unique-Filename Dataset Split Completed ===")
print(f"Successfully split {segmented_count} logo classes.")
print(f"Total unique paired files copied to disk: {total_copied}")

=== Unique-Filename Dataset Split Completed ===
Successfully split 371 logo classes.
Total unique paired files copied to disk: 15533


In [84]:
import os
import glob

# Check the physical count on disk now
print("=== NEW DISK STATS ===")
for split in ['train', 'val', 'test']:
    imgs = os.listdir(f"/content/logo_yolo_dataset/images/{split}")
    lbls = os.listdir(f"/content/logo_yolo_dataset/labels/{split}")
    print(f"{split.capitalize()} Images: {len(imgs)} | Labels: {len(lbls)}")

print("\n=== Checking TeamViewer labels in train split ===")
tv_train_files = glob.glob("/content/logo_yolo_dataset/labels/train/teamviewer_*.txt")
print(f"Found {len(tv_train_files)} unique TeamViewer training files physically written to disk!")
if tv_train_files:
    with open(tv_train_files[0], 'r') as f:
        print(f"Sample label content from '{os.path.basename(tv_train_files[0])}':\n{f.read().strip()}")

=== NEW DISK STATS ===
Train Images: 12257 | Labels: 12257
Val Images: 1558 | Labels: 1558
Test Images: 1718 | Labels: 1718

=== Checking TeamViewer labels in train split ===
Found 71 unique TeamViewer training files physically written to disk!
Sample label content from 'teamviewer_6.txt':
306 0.585239 0.478643 0.754678 0.414573
306 0.113306 0.581658 0.189189 0.670854


### Step 6: Generate Dataset Configuration YAML

In [85]:
# Generate dataset configuration YAML using the EXACT same deterministic mapping from Step 4
class_dict = {i: name for name, i in class_to_id.items()}

yaml_data = {
    'path': DATASET_ROOT,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': class_dict
}

yaml_path = os.path.join(DATASET_ROOT, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"YOLO dataset configuration file saved to: {yaml_path}")

YOLO dataset configuration file saved to: /content/logo_yolo_dataset/dataset.yaml


### Step 7: Train the YOLO Detector

Optionally clear the existing model weights:

In [86]:
import shutil
import os

target_dir = '/content/runs/detect/logo_detection/yolov8_logos'

if os.path.exists(target_dir):
    # Forcefully delete the directory and all files inside
    shutil.rmtree(target_dir)
    print(f"Successfully force-deleted: {target_dir}")
else:
    print(f"Directory does not exist: {target_dir}")

Successfully force-deleted: /content/runs/detect/logo_detection/yolov8_logos


In [ ]:
print("Initializing pre-trained YOLOv8 nano model...")
model = YOLO('yolov8n.pt')

# Train the model with checkpoint saving enabled
# By default, YOLOv8 saves checkpoints in {project}/{name}/weights/last.pt and best.pt
results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,          # Set to 0 if utilizing a GPU runtime
    save=True,         # This guarantees that checkpoint weights (.pt files) are saved
    project='logo_detection',
    name='yolov8_logos'
)

print("Training process finished!")

# --- HOW TO RESUME FROM A CHECKPOINT ---
# If your training is interrupted, you can resume it by uncommenting and running the code below:
# model = YOLO('/content/logo_detection/yolov8_logos/weights/last.pt')
# model.train(resume=True)

Initializing pre-trained YOLOv8 nano model...
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/logo_yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yol

### Step 8: Inference on Custom Test Images
This section sets up a local test directory where you can upload your own custom images. It runs the trained YOLO model on those images, uses OpenCV (`cv2`) to draw bounding boxes and class labels directly on the images, and outputs both the visual image results and the corresponding bounding box coordinate `.txt` files.

In [17]:
import cv2
from google.colab import files

# --- CONFIGURATION ---
TEST_UPLOAD_DIR = "/content/custom_test_images"
INFERENCE_OUTPUT_DIR = "/content/custom_test_results"

# Prepare folders
os.makedirs(TEST_UPLOAD_DIR, exist_ok=True)
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

print(f"Test upload directory is ready at: {TEST_UPLOAD_DIR}")
print(f"Inference results will be saved to: {INFERENCE_OUTPUT_DIR}")

Test upload directory is ready at: /content/custom_test_images
Inference results will be saved to: /content/custom_test_results


#### Run this cell to upload your custom logo images to the test folder:

In [19]:
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(TEST_UPLOAD_DIR, filename))
print(f"Uploaded {len(uploaded)} images to {TEST_UPLOAD_DIR}")

Saving TeamViewer-logo.png to TeamViewer-logo.png
Uploaded 1 images to /content/custom_test_images


#### Run the detector, draw bounding boxes, and save outputs:

In [68]:
# 1. Load the best trained model (using trained weights if available, fallback to basic nano)
model_path = '/content/logo_detection/yolov8_logos/weights/best.pt'
if os.path.exists(model_path):
    test_model = YOLO(model_path)
    print(f"Loaded custom trained model weights from: {model_path}")
else:
    test_model = model
    print("Using the currently active trained model instance.")

# 2. Get all images in test folder
test_images = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.JPG', '*.JPEG', '*.PNG'):
    test_images.extend(glob.glob(os.path.join(TEST_UPLOAD_DIR, ext)))

if len(test_images) == 0:
    print(f"No images found in {TEST_UPLOAD_DIR}. Please upload some files in the step above.")
else:
    # 3. Perform prediction and annotate
    for img_path in test_images:
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        # Load image with OpenCV
        img = cv2.imread(img_path)
        h, w, _ = img.shape

        # Run YOLO inference with a lowered confidence threshold of 0.10 to help catch detections
        results = test_model.predict(source=img_path, conf=0.10, verbose=False)

        txt_output_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{base_name}_predictions.txt")
        detections_found = []

        with open(txt_output_path, 'w') as txt_file:
            for r in results:
                boxes = r.boxes
                for box in boxes:
                    # Get coordinate values
                    coords = box.xyxy[0].tolist() # Left, Top, Right, Bottom
                    cls_id = int(box.cls[0].item())
                    conf = float(box.conf[0].item())
                    cls_name = test_model.names.get(cls_id, f"Class_{cls_id}")

                    detections_found.append((cls_name, conf))
                    x1, y1, x2, y2 = map(int, coords)

                    # Save coordinates in normalized format to the .txt file: [class_id center_x center_y width height]
                    dw = 1.0 / w
                    dh = 1.0 / h
                    cx = (x1 + x2) / 2.0 * dw
                    cy = (y1 + y2) / 2.0 * dh
                    bw = (x2 - x1) * dw
                    bh = (y2 - y1) * dh
                    txt_file.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

                    # Draw bounding box on the image using OpenCV
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

                    # Add class and confidence label
                    label_str = f"{cls_name}: {conf:.2f}"
                    cv2.putText(img, label_str, (x1, max(y1 - 10, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Save the annotated image
        annotated_img_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{base_name}_result.jpg")
        cv2.imwrite(annotated_img_path, img)

        if len(detections_found) > 0:
            print(f"\nProcessed {os.path.basename(img_path)} -> Detections saved to {INFERENCE_OUTPUT_DIR}")
            print("Detections:")
            for cls_name, conf in detections_found:
                print(f" - {cls_name}: {conf:.4f} confidence")
        else:
            print(f"Processed {os.path.basename(img_path)} -> No logos detected above confidence threshold 0.10. {txt_output_path} is empty.")

Using the currently active trained model instance.
Processed TeamViewer-logo.png -> No logos detected above confidence threshold 0.10. /content/custom_test_results/TeamViewer-logo_predictions.txt is empty.
Processed 2.jpg -> No logos detected above confidence threshold 0.10. /content/custom_test_results/2_predictions.txt is empty.

Processed 4.jpg -> Detections saved to /content/custom_test_results
Detections:
 - basilisk: 0.1684 confidence

Processed 5.jpg -> Detections saved to /content/custom_test_results
Detections:
 - baird: 0.3194 confidence
 - new holland-3: 0.1263 confidence
Processed 1.jpg -> No logos detected above confidence threshold 0.10. /content/custom_test_results/1_predictions.txt is empty.


### Step 9: Diagnostic Check - Inspect 'teamviewer' Dataset Integrity
Run this cell to count how many images and label files actually exist for `teamviewer` in your source directory, and see what the bounding box files contain.

In [81]:
import os
import glob

# --- Unified Single Dataset Validation Cell (Diagnostic Mode) ---
TARGET_BRAND = "teamviewer"
print(f"=== VALIDATING DATASET INTEGRITY FOR '{TARGET_BRAND}' ===\n")

# 1. Resolve Expected Class ID from deterministic map
expected_id = class_to_id.get(TARGET_BRAND, 306) if 'class_to_id' in globals() else 306
print(f"-> Expected Class ID in dataset.yaml: {expected_id}\n")

# 2. Let's inspect the files on disk vs the labels inside them
DATASET_ROOT = "/content/logo_yolo_dataset"

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_ROOT, 'images', split)
    lbl_dir = os.path.join(DATASET_ROOT, 'labels', split)

    # Count physical files copied by Step 5
    physical_label_files = []
    if os.path.exists(lbl_dir):
        # Filter files that belong to teamviewer (since we know teamviewer file names from original folder)
        # Let's inspect the actual lines written inside whatever label files we have in this split
        physical_label_files = glob.glob(os.path.join(lbl_dir, "*.txt"))

    print(f"--- Split '{split}': ---")
    print(f"  * Total physical label files (.txt) in this split folder: {len(physical_label_files)}")

    # Read what class IDs are actually written inside these labels
    found_ids_in_split = set()
    matching_expected_id_count = 0
    sample_lines = []

    for txt_file in physical_label_files:
        with open(txt_file, 'r') as f:
            content = f.read().strip()
            if content:
                for line in content.split('\n'):
                    parts = line.split()
                    if parts:
                        cls_id = parts[0]
                        found_ids_in_split.add(cls_id)
                        if cls_id == str(expected_id):
                            matching_expected_id_count += 1
                            if len(sample_lines) < 3:
                                sample_lines.append((os.path.basename(txt_file), line))

    print(f"  * Distinct Class IDs actually written inside these text files: {sorted(list(found_ids_in_split))[:15]}")
    print(f"  * Files containing the target class ID '{expected_id}': {matching_expected_id_count}")
    if sample_lines:
         print("  * Sample matching lines:")
         for fname, s_line in sample_lines:
              print(f"     - {fname}: {s_line}")
    print("")

=== VALIDATING DATASET INTEGRITY FOR 'teamviewer' ===

-> Expected Class ID in dataset.yaml: 306

--- Split 'train': ---
  * Total physical label files (.txt) in this split folder: 12257
  * Distinct Class IDs actually written inside these text files: ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110']
  * Files containing the target class ID '306': 108
  * Sample matching lines:
     - teamviewer_6.txt: 306 0.585239 0.478643 0.754678 0.414573
     - teamviewer_6.txt: 306 0.113306 0.581658 0.189189 0.670854
     - teamviewer_72.txt: 306 0.446629 0.736626 0.449438 0.078189

--- Split 'val': ---
  * Total physical label files (.txt) in this split folder: 1558
  * Distinct Class IDs actually written inside these text files: ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110']
  * Files containing the target class ID '306': 13
  * Sample matching lines:
     - teamviewer_22.txt: 306 0.635189 0.47

### Step 9: Diagnostic Check - Inspect Training Results & Validate Model
Run this cell to find the exact path of your newly trained weights and run a standard validation batch. This verifies if the model actually learned anything during those 5 epochs.

In [22]:
import glob
import os
from ultralytics import YOLO

# 1. Automatically find the latest trained weights under logo_detection directory
weights_paths = glob.glob("/content/runs/detect/logo_detection/*/weights/best.pt") + glob.glob("/content/logo_detection/*/weights/best.pt")

if weights_paths:
    # Sort by modification time to get the newest run's weights
    latest_weights = max(weights_paths, key=os.path.getmtime)
    print(f"[SUCCESS] Found trained weights at: {latest_weights}")

    # Load the trained model
    diagnostic_model = YOLO(latest_weights)

    # 2. Run standard validation to see overall model performance metrics (mAP50, Precision, Recall)
    print("\n--- Running standard validation on validation dataset split ---")
    val_results = diagnostic_model.val()
else:
    print("[WARNING] No trained 'best.pt' weights found anywhere! Your model has not saved training weights successfully or the path is wrong.")

[SUCCESS] Found trained weights at: /content/runs/detect/logo_detection/yolov8_logos/weights/best.pt

--- Running standard validation on validation dataset split ---
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,422,837 parameters, 0 gradients, 10.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 811.0±363.1 MB/s, size: 15.4 KB)
val: Scanning /content/logo_yolo_dataset/labels/val.cache... 202 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 202/202 56.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 5.1it/s 2.6s
                   all        202        256          0          0          0          0
             7Days Inn          1          1          0          0          0          0
                 Amoco          2          2          0          0          0          0
                 Ampeg          1          1        